# 5.1 Balance tables

This notebook does the following:
    - Balance table comparing control vs treatment

## Set-up

In [ ]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
import importlib
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import os
import statsmodels.formula.api as smf
import pyfixest as pf

sys.path.append(str(Path.cwd().parents[0] / "functions"))
from make_balance_table import make_balance_table

In [2]:
# Load data
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv", 
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

## (1) Prepare data for balance tables

In [3]:
# Construct post variable
df["post"] = (df["survey"] == "EL").astype(int)

In [4]:
# Keep only labgroups that have both pre and post observations
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
labgroups_to_keep = labgroup_counts[labgroup_counts == 2].index
df = df[df["labgroupid"].isin(labgroups_to_keep)].copy()

In [5]:
# Keep only BL observations for balance table
df = df[df["survey"] == "BL"].copy()

In [6]:
# Science faculty indicator
df["science_faculty"] = (df["faculty"] == "Faculty of Science (MNF)").astype(int)

# Missing survey indicator
df["missing_bl_date"] = (df["survey_date_bl"] == "").astype(int)

# Sharing equipment and space indicators
for var in ["share_equip", "share_space"]:
    df[var] = (df[f"{var}_ind"] == "Yes").astype(int)

# Waste categories
for var in ["waste_recycle", "waste_clinical", "waste_general"]:
    df[f"{var}_over3"] = df[var].isin(["3-6kg", "6-10kg", "10-12kg", ">12kg"]).astype(int)

# Specialized equipment indicators
for var in [
    "pcr", 
    "ice", 
    "centrifuge", 
    "coffee", 
    "microwave", 
    "animal", 
    "nonco2_incubator", 
    "4c_room", 
    "minus_20c_room",
    "other"
    ]:
    df[f"{var}_indicator"] = (df[f"{var}_ind"] == "Yes").astype(int)

# Any specialized equipment indicator
df["any_spec_indicator"] = df[[f"{var}_indicator" for var in [
    "pcr", 
    "ice", 
    "centrifuge", 
    "coffee", 
    "microwave", 
    "animal", 
    "nonco2_incubator", 
    "4c_room", 
    "minus_20c_room",
    "other"
]]].any(axis=1).astype(int)

In [10]:
# Variable labels
var_labels = {
    "science_faculty": "Science Faculty",
    "no_researchers": "Number of researchers",
    "no_ft": "Number of full-time researchers",
    "annual_electricity_total": "Total",
    "annual_electricity_fc": "Fume cupboards",
    "annual_electricity_fridge": "Fridges",
    "annual_electricity_freezer": "Freezers",
    "annual_electricity_ult": "ULT freezers",
    "annual_electricity_cryostat": "Cryostats",
    "annual_electricity_microbio": "Microbiological safety cabinets",
    "annual_electricity_incubator": "CO2 incubators",
    "annual_electricity_glassware": "Glassware drying cabinets",
    "annual_electricity_bath": "Water baths",
    "annual_electricity_heater": "Block heaters",
    "annual_electricity_it": "IT equipment",
    "share_equip": "Share any equipment",
    "share_space": "Share any space",
    "waste_recycle_over3": "Recycling waste $>$ 3kg",
    "waste_clinical_over3": "Clinical waste $>$ 3kg",
    "waste_general_over3": "General waste $>$ 3kg",
    "any_spec_indicator": "Any specialized equipment",
    "pcr_indicator": "PCR machine",
    "ice_indicator": "Ice machine",
    "centrifuge_indicator": "Centrifuge",
    "coffee_indicator": "Coffee machine",
    "microwave_indicator": "Microwave",
    "animal_indicator": "Animal facility",
    "nonco2_incubator_indicator": "Non-CO2 incubator",
    "4c_room_indicator": "4\degree C room",
    "minus_20c_room_indicator": "-20\degree C room"
}

## (2) Balance table


In [11]:
# Headers in balance table
section_headers = {
    "Baseline annual electricity consumption (kWh)": [
        "annual_electricity_total",
        "annual_electricity_fc",
        "annual_electricity_fridge",
        "annual_electricity_freezer",
        "annual_electricity_ult",
        "annual_electricity_cryostat",
        "annual_electricity_microbio",
        "annual_electricity_incubator",
        "annual_electricity_glassware",
        "annual_electricity_bath",
        "annual_electricity_heater",
        "annual_electricity_it"]}

# Annual electricity vars get 1 decimal places; everything else falls back to default_decimals
one_dec_vars = section_headers["Baseline annual electricity consumption (kWh)"]
decimals = {var: 1 for var in one_dec_vars}

# Number researchers and FT get 2 decimal places
decimals.update({var: 2 for var in ["no_researchers", "no_ft"]})

# Create balance table
table = make_balance_table(
    df = df,
    treatment_var = "treated",
    var_labels    = var_labels,
    section_headers = section_headers,
    control_val = 0,
    treatment_val = 1,
    control_label = "Control",
    treatment_label = "Treatment",
    decimals      = decimals,
    default_decimals = 3,
    col1_width    = "9cm",
    coln_width    = "2cm",
    indent=r"\hspace{0.3cm} ",
)
table_path = config.OUTPUT / "7_Balance_Tables" / "balance_tab_pooled.tex"
_ = table_path.write_text(table)

In [9]:
# Check that share_equip is indeed unbalanced between T and C
check = smf.ols("share_equip ~ treated", data=df).fit(cov_type="HC1")
print(check.summary())

                            OLS Regression Results                            
Dep. Variable:            share_equip   R-squared:                       0.072
Model:                            OLS   Adj. R-squared:                  0.064
Method:                 Least Squares   F-statistic:                     8.508
Date:                Fri, 17 Jul 2026   Prob (F-statistic):            0.00431
Time:                        15:07:36   Log-Likelihood:                -56.129
No. Observations:                 109   AIC:                             116.3
Df Residuals:                     107   BIC:                             121.6
Df Model:                           1                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.8868      0.044     20.188      0.0

## (3) Balance tables by faculty

In [13]:
# Variables to include (everything in var labels except "science_faculty")
vars_to_include = [var for var in var_labels.keys() if var != "science_faculty"]
var_labels_by_faculty = {var: var_labels[var] for var in vars_to_include}

In [14]:
# Define dfs for science faculty and non-science faculty
df_science = df[df["science_faculty"] == 1].copy()
df_non_science = df[df["science_faculty"] == 0].copy()

In [15]:
# Headers in balance table
section_headers = {
    "Baseline annual electricity consumption (kWh)": [
        "annual_electricity_total",
        "annual_electricity_fc",
        "annual_electricity_fridge",
        "annual_electricity_freezer",
        "annual_electricity_ult",
        "annual_electricity_cryostat",
        "annual_electricity_microbio",
        "annual_electricity_incubator",
        "annual_electricity_glassware",
        "annual_electricity_bath",
        "annual_electricity_heater",
        "annual_electricity_it"]}

# Annual electricity vars get 1 decimal places; everything else falls back to default_decimals
one_dec_vars = section_headers["Baseline annual electricity consumption (kWh)"]
decimals = {var: 1 for var in one_dec_vars}

# Number researchers and FT get 2 decimal places
decimals.update({var: 2 for var in ["no_researchers", "no_ft"]})

# Create balance table for science faculty and non science faculty
for faculty in ["science", "non_science"]:

    if faculty == "science":
        df_faculty = df_science
    if faculty == "non_science":
        df_faculty = df_non_science

    table = make_balance_table(
        df = df_faculty,
        treatment_var = "treated",
        var_labels    = var_labels_by_faculty,
        section_headers = section_headers,
        control_val = 0,
        treatment_val = 1,
        control_label = "Control",
        treatment_label = "Treatment",
        decimals      = decimals,
        default_decimals = 3,
        col1_width    = "9cm",
        coln_width    = "2cm",
        indent=r"\hspace{0.3cm} ",
    )
    table_path = config.OUTPUT / "7_Balance_Tables" / f"balance_tab_{faculty}.tex"
    _ = table_path.write_text(table)